# 02. Схема валидации и baseline

**Валидация** (`src/validation.py`) повторяет ситуацию теста на прошлых датах. На 15.12.2025 впереди
17 матчей, это весь остаток сезона, и на 6 из них продажи уже открыты. Для каждой даты «псевдо-сегодня» `as_of`:
- **train**: матчи, сыгранные до `as_of`, из `build_table(as_of)`;
- **val**: следующие до 17 матчей того же сезона. Признаки берутся из `build_table(as_of)`, итог из `build_table()`;
- прогноз обрезается по правилам сдачи: `sold_as_of ≤ ŷ ≤ capacity`, целое.

**Baseline**: среднее `tickets_total` по зоне на train фолда.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
from IPython.display import Markdown, display

from src.data import build_table
from src.validation import FOLDS, evaluate, make_fold, predict_test

pd.set_option("display.precision", 1)

## Фолды

In [2]:
full = build_table()
rows = []
for as_of in FOLDS:
    train, val, y = make_fold(as_of, full)
    ids = val["match_id"].unique()
    last = train["match_date"].max()
    rows.append({
        "as_of": as_of,
        "train_matches": train["match_id"].nunique(),
        "val": f"{ids[0]}–{ids[-1]}",
        "val_matches": len(ids),
        "val_in_sale": val.groupby("match_id")["sales_started"].max().sum(),
        "days_since_last_game": (pd.Timestamp(as_of) - last).days,
    })
test = full[full["is_test"] == 1]
rows.append({"as_of": "2025-12-15 (test)", "train_matches": full["tickets_total"].notna().sum() // 8,
             "val": "M086–M102", "val_matches": test["match_id"].nunique(),
             "val_in_sale": test.groupby("match_id")["sales_started"].max().sum(),
             "days_since_last_game": (pd.Timestamp("2025-12-15") - full.loc[full["is_test"] == 0, "match_date"].max()).days})
pd.DataFrame(rows).set_index("as_of")

,train_matches,val,val_matches,val_in_sale,days_since_last_game
as_of,,,,,
2023-12-15,20,M021–M034,14,5,10
2024-11-25,51,M052–M068,17,5,3
2025-09-20,73,M074–M085,12,3,3
2025-12-15 (test),85,M086–M102,17,6,8


## Baseline: среднее по зоне

In [3]:
def baseline_predict(train, val, as_of):
    zone_mean = train.groupby("zone")["tickets_total"].mean()
    return val["zone"].map(zone_mean).to_numpy()


res = evaluate(baseline_predict)
res

EvalResult(RMSE=141.04, folds=['2023-12-15', '2024-11-25', '2025-09-20'])

### RMSE по фолдам

In [4]:
res.by_fold

,rmse,matches,rows
fold,,,
2023-12-15,172.2,14,112
2024-11-25,103.0,17,136
2025-09-20,147.2,12,96
all,141.0,43,344


### RMSE по зонам

In [5]:
res.by_zone

fold,2023-12-15,2024-11-25,2025-09-20,all
zone,,,,
1,46.4,45.4,27.8,41.6
2,40.4,28.5,47.1,38.4
3,63.6,57.0,71.0,63.3
4,220.1,117.2,144.2,164.3
5,130.1,110.0,77.6,109.4
6,286.4,204.3,286.1,257.0
7,282.8,86.4,227.8,208.5
8,44.7,61.6,68.3,58.8


### RMSE: матчи, уже открытые к продаже на as_of, и ещё не открытые

In [6]:
res.by_status

,rmse,matches,rows
in_sale,,,
ещё не в продаже,149.5,30,240
в продаже,119.3,13,104


In [7]:
display(Markdown(f"# Baseline RMSE = {res.overall:.1f}"))

# Baseline RMSE = 141.0

## Где baseline ошибается

In [8]:
p = res.preds.assign(err=lambda d: d["y_pred"] - d["y_true"])
by_match = (p.groupby(["fold", "match_id"])
              .agg(y_true=("y_true", "sum"), y_pred=("y_pred", "sum"), in_sale=("in_sale", "max"))
              .assign(err=lambda d: d["y_pred"] - d["y_true"])
              .merge(full.drop_duplicates("match_id").set_index("match_id")[["match_date", "opponent", "weekday"]],
                     left_on="match_id", right_index=True))
by_match.reindex(by_match["err"].abs().sort_values(ascending=False).index).head(10)

y_true  y_pred  in_sale   err match_date      opponent  \
fold       match_id                                                           
2025-09-20 M079        5691    4608    False -1083 2025-10-27  Металлург Мг   
2023-12-15 M022        3802    4708     True   906 2023-12-21          Сочи   
2025-09-20 M075        5649    4757     True  -892 2025-10-03     Локомотив   
           M076        5449    4608     True  -841 2025-10-06  Автомобилист   
2024-11-25 M053        3846    4642     True   796 2024-12-04          Сочи   
2023-12-15 M034        5301    4708    False  -593 2024-02-22          Лада   
2025-09-20 M084        4033    4608    False   575 2025-12-04          Сочи   
2023-12-15 M021        4252    4827     True   575 2023-12-19       Ак Барс   
2025-09-20 M080        5165    4608    False  -557 2025-10-29         Барыс   
           M081        5140    4608    False  -532 2025-11-01    Нефтехимик   

                     weekday  
fold       match_id           
2025-09-20 M079            1  
2023-12-15 M022            4  
2025-09-20 M075            5  
           M076            1  
2024-11-25 M053            3  
2023-12-15 M034            4  
2025-09-20 M084            4  
2023-12-15 M021            2  
2025-09-20 M080            3  
           M081            6

In [9]:
# Средняя ошибка (bias) по зонам: > 0 — baseline завышает
p.pivot_table(index="zone", columns="fold", values="err", aggfunc="mean")

fold,2023-12-15,2024-11-25,2025-09-20
zone,,,
1,25.9,-33.6,4.8
2,20.7,24.3,43.8
3,34.6,15.6,44.5
4,-150.8,-103.4,-139.1
5,-84.5,-29.6,-63.3
6,-92.7,200.2,-210.6
7,118.4,-41.4,-129.6
8,-25.4,-35.8,-2.0


**Выводы по ошибкам baseline:**
- Почти вся ошибка приходится на массовые зоны 4, 6 и 7: RMSE 160–260 против 40–65 в зонах 1–3 и 8.
- Смещение по зонам меняет знак от фолда к фолду. В зоне 6 baseline завышает на 200 в 2024/25 и занижает на 210 в 2025/26. Уровень спроса в зоне плывёт между сезонами, а общее среднее этого не видит. Стоит проверить, как меняются `capacity`/`season_tickets` и насколько полезно взвешивать недавние матчи.
- В сезоне 2025/26 baseline системно занижает: M075, M076, M079–M081 недопрогнозированы на 500–1100 билетов за матч. Спрос текущего сезона выше исторического среднего.
- Baseline одинаков для всех матчей. Различия по сопернику, дню недели и дате не учитываются, отсюда ошибки ±600–900 на отдельных матчах (M022, M053).
- По матчам, уже открытым к продаже, RMSE ниже (119 против 149) отчасти за счёт нижней границы `sold_as_of`. Темп уже начавшихся продаж должен стать сильным признаком.
- В первом фолде (2023-12-15) train всего 20 матчей, поэтому его RMSE заметно шумнее остальных.

## Прогноз на test

In [10]:
sub = predict_test(baseline_predict)
assert len(sub) == 136 and (sub["tickets_total"] >= 0).all()
assert sub["tickets_total"].dtype.kind == "i"

out = Path("../submissions")
out.mkdir(exist_ok=True)
sub.to_csv(out / "baseline.csv", index=False, encoding="utf-8")
sub.pivot(index="match_id", columns="zone", values="tickets_total")

zone,1,2,3,4,5,6,7,8
match_id,,,,,,,,
M086,465,308,633,782,767,487,781,450
M087,465,308,633,782,767,700,781,450
M088,465,308,633,782,767,518,781,450
M089,465,308,633,782,767,621,781,450
M090,465,308,633,782,767,487,781,450
M091,465,308,633,782,767,487,781,450
M092,465,308,633,782,767,487,781,450
M093,465,308,633,782,767,487,781,450
M094,465,308,633,782,767,487,781,450
